Add a conversion from zipcodes to tract_id's

In [1]:
import geopandas as gpd
import pandas as pd

# Download ZIP code tabulation area (ZCTA) shapefile for California
import urllib.request
url = "https://www2.census.gov/geo/tiger/TIGER2010/ZCTA5/2010/tl_2010_06_zcta510.zip"
urllib.request.urlretrieve(url, "ca_zcta.zip")

# Load both shapefiles
zcta = gpd.read_file("zip://ca_zcta.zip")
tracts = gpd.read_file("bay_area_tracts.shp")

# Standardize CRS
zcta = zcta.to_crs("EPSG:3310")
tracts = tracts.to_crs("EPSG:3310")

# Spatial join — assign each tract to the ZCTA it overlaps most with
crosswalk = gpd.sjoin(tracts, zcta[["ZCTA5CE10", "geometry"]], how="left", predicate="intersects")
crosswalk = crosswalk.rename(columns={"ZCTA5CE10": "zip_code"})

# Standardize tract ID
crosswalk.columns = [c.replace("00","").replace("10","") for c in crosswalk.columns]
if "CTIDFP" in crosswalk.columns:
    crosswalk = crosswalk.rename(columns={"CTIDFP": "tract_id"})

crosswalk = crosswalk[["tract_id", "zip_code"]].drop_duplicates(subset="tract_id")
crosswalk["tract_id"] = crosswalk["tract_id"].astype(str).str.zfill(11)

print(crosswalk.head())
print(f"Crosswalk built: {len(crosswalk)} tracts mapped to zip codes")
crosswalk.to_csv("zip_tract_crosswalk.csv", index=False)

      tract_id zip_code
0  06059062629    92603
1  06059062630    92603
2  06059062631    92603
3  06059062632    92656
4  06059062633    92656
Crosswalk built: 7049 tracts mapped to zip codes


In [2]:
from google.colab import files
files.download('zip_tract_crosswalk.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>